## What This Notebook Does
This notebook generates Accumulated Local Effects (ALE) explanations for the trained spatial models in this repository.

It runs one ALE variant (`main`, `pairwise`, or `full`) across selected prediction types, then saves both per-run metadata and a bundled result object for downstream analysis.

### Before You Run
- Build the `preles_c` extension (README, section 2).
- Put the trained models under `checkpoints/`, or set `CHECKPOINT_PATH`. The expected layout is
  `checkpoints/preles/full/station_<length>/logs_spatial_<model type>/outer_station_<site>/`.
- The PRELES calibrations for the models' training splits must exist under `Data/calibrations/`.
- Mixture-of-experts models are not covered here.


### 1) Imports and Variant Registry
This cell loads project utilities and maps each variant name to the corresponding ALE generation function.

- `main`: standard 1D ALE for top models: first-order calculations predictor x target
- `pairwise`: 2D pairwise ALE focused on top models: 2nd-order calculations predictor x predictor - only the interaction strength!
- `full`: broader pairwise output for top models: 2nd-order calculations predictor x predictor interaction effect + main effect

`PROJECT_ROOT` is the repository root; the notebook changes to it, so start it from `XAI/`.


In [ ]:
# --- Imports and path setup ---
import os
import sys
import json
import glob
import warnings

import joblib

warnings.filterwarnings("ignore", message="Series.ravel is deprecated", category=FutureWarning)

PROJECT_ROOT = os.getenv("PROJECT_ROOT") or os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)   # data and calibration paths are relative to the repository root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from XAI.xai_utils.ale_registry import VARIANT_FN
from XAI.xai_utils.batch_utils import batch_explain


### 2) User Configuration
Edit this section to control what gets explained and where outputs are written.

Key settings:
- `variant`: ALE mode (`main`, `pairwise`, `full`).
- `length`: number of training years in the CV design (`len3`, `len6`, `len9`, `len12`).
- `prediction_types`: model types to process.
- `grid_size`: ALE bin resolution (higher = finer but slower).

`CHECKPOINT_PATH`, `ALE_VARIANT`, `ALE_LENGTH`, `ALE_GRID_SIZE`, `ALE_CHAIN_PROCESS` and `ALE_PAIRS` (e.g. `PAR_VPD`) can also be set in the environment.

`chain_process=True` (`main` and `full`) re-runs PRELES for each perturbed day before the network, so the curves show the hybrid's full response to the weather; it needs `residual`, `bias` or `parallel` models and results go to `<variant>_chain`. With `chain_process=False` PRELES is held fixed, which for `parallel` means the curves show the network part only.

For `pairwise`, `pairwise_impute_empty_cells=True` fills empty 2D bins with nearest-neighbor values, which improves plot continuity but can introduce combinations not present in observed data, or not physically plausible.


In [ ]:
# --- CONFIG (EDIT HERE) ---
variant = os.getenv("ALE_VARIANT", "main")      # {"main", "pairwise", "full"}
length = os.getenv("ALE_LENGTH", "len12")       # {"len3", "len6", "len9", "len12"}
prediction_types = ["naive", "bias", "parallel", "regularized", "residual"]

# False: PRELES outputs held fixed, ALE of the network alone.
# True ("main" and "full" only): each perturbed day is re-run through PRELES, then the network, so the
# curves show the full response to the weather. Hybrids only; naive and regularized do not use PRELES.
chain_process = os.getenv("ALE_CHAIN_PROCESS", "").lower() in {"1", "true", "yes"}
# "full" only: restrict to some feature pairs, e.g. [["PAR", "VPD"]]; None = all pairs
feature_pairs = [p.strip().split("_", 1) for p in os.getenv("ALE_PAIRS", "").split(",") if p.strip()] or None
if chain_process:
    prediction_types = [t for t in prediction_types if t in {"residual", "bias", "parallel"}]

grid_size = int(os.getenv("ALE_GRID_SIZE", 20)) # 10 coarser and quicker, 30 finer and slower
checkpoint_every = 1
save_intermediate = True
overwrite = False                               # False resumes from the per-type caches
pairwise_impute_empty_cells = True              # fills empty 2D bins with nearest-neighbour values;
                                                # can introduce combinations that do not occur in the data

# Paths
checkpoint_root = os.getenv("CHECKPOINT_PATH", os.path.join(PROJECT_ROOT, "checkpoints"))
logs_root = os.path.join(checkpoint_root, "preles", "full", f"station_{length}")
cv_config_path = os.path.join(PROJECT_ROOT, "configs", "CV_splits", "Paper_1_preles_ncvs",
                              f"CV_station_{length}_config.json")
save_dir_base = os.path.join(PROJECT_ROOT, "XAI", "results", "ALE", "paper1_spatial")
variant_dir = f"{variant}_chain" if chain_process else variant
root = os.path.join(save_dir_base, f"spatial_{length}", f"grid{grid_size}", variant_dir)
bundle_path = os.path.join(root, f"{variant}_bundle.joblib")

print("[CONFIG] variant:", variant)
print("[CONFIG] length:", length)
print("[CONFIG] prediction_types:", prediction_types)
print("[CONFIG] chain_process:", chain_process)
print("[CONFIG] feature_pairs:", feature_pairs or "all")
print("[CONFIG] logs_root:", logs_root)
print("[CONFIG] cv_config_path:", cv_config_path)
print("[CONFIG] output folder:", root)
print("[CONFIG] grid_size:", grid_size)
print("[CONFIG] overwrite:", overwrite)
if variant == "pairwise":
    print("[CONFIG] pairwise_impute_empty_cells:", pairwise_impute_empty_cells)
if chain_process and variant not in {"main", "full"}:
    raise ValueError("chain_process is only implemented for the 'main' and 'full' variants.")
if feature_pairs and variant != "full":
    raise ValueError("feature_pairs is only implemented for the 'full' variant.")

# --- Load CV config ---
with open(cv_config_path, "r") as f:
    CV_CONFIG = json.load(f)

# --- Preflight: warn if any prediction type has no matching logs ---
for pred_type in prediction_types:
    logs_glob = os.path.join(logs_root, f"logs_spatial_{pred_type}*")
    if not glob.glob(logs_glob):
        print(f"[WARNING] No log directories found for prediction type '{pred_type}' (pattern: {logs_glob})")


# Runtime warning
ALE calculations can take a long time across prediction types, outer folds, and feature grids.
Typical runs can take from ~30 minutes to a few hours depending on grid size and model count.
The code is designed to save intermediate results, so if you stop and restart it will continue on with the most recent prediction type not completed.


## 3) Run ALE Batch and Save Artifacts
This final cell:
- infers the NCV kind from logs,
- creates output folders,
- runs ALE generation in batch mode for `GPP` and `ET`,
- writes a `*.joblib` bundle with all results,
- writes a `*_meta.json` file with run settings.
- intermediate results are saved to minimise on rerun time should errors occur


Expected outputs in the run folder:
- `<variant>_bundle.joblib`: serialized ALE results.
- `<variant>_meta.json`: reproducibility metadata.
- `intermediate/` (optional): checkpointed partial outputs.


In [ ]:
# --- Run ALE batch ---
save_intermediate_dir = os.path.join(root, "intermediate") if save_intermediate else None
os.makedirs(save_intermediate_dir or root, exist_ok=True)

batch_kwargs = dict(
    grid_size=grid_size,
    save_intermediate_dir=save_intermediate_dir,
    checkpoint_every=checkpoint_every if checkpoint_every else 0,
    overwrite=bool(overwrite),
)
if variant == "pairwise":
    batch_kwargs["impute_empty_cells"] = pairwise_impute_empty_cells
if chain_process:
    batch_kwargs["chain_process"] = True
if feature_pairs:
    batch_kwargs["feature_pairs"] = feature_pairs

results = batch_explain(
    explanation_fn=VARIANT_FN[variant],
    prediction_types=prediction_types,
    logs_root=logs_root,
    cv_config=CV_CONFIG,
    save_dir=root,
    file_prefix=variant,
    target_list=["GPP", "ET"],
    use_process=True,
    **batch_kwargs,
)

# Normalisation constants per fold, to convert the z-scored ALE x-axes back to physical units.
norm_stats = results.pop("normalization_stats", {})
if norm_stats:
    with open(os.path.join(root, f"{variant}_normalization_stats.json"), "w") as f:
        json.dump(norm_stats, f, indent=2)

joblib.dump(results, bundle_path)

meta = dict(
    variant=variant, ncv_kind="spatial",
    length=length, grid_size=grid_size,
    prediction_types=prediction_types,
    chain_process=chain_process,
    feature_pairs=feature_pairs,
    logs_root=logs_root,
    save_dir_base=save_dir_base,
    cv_config_path=cv_config_path,
)
if variant == "full":
    meta["full_effect"] = "main+interaction"
with open(os.path.join(root, f"{variant}_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"[OK] Saved ALE to: {bundle_path}")
print(f"[OK] Per-type caches are under: {root}")


## Interpreting and Reusing Results
- Use the `*_meta.json` file to confirm the exact configuration used for a run.
- Load `*_bundle.joblib` in follow-up notebooks/scripts for plotting or comparison across variants.
- If results are unexpectedly sparse, first re-check `prediction_types`, `logs_root`, and CV config path consistency.


## 6) Plot ALE Results

### Available Plot Modes
**For `main` (1D ALE):**
- `station_vs_prediction` (default): grid of line plots for each feature, split by output (`GPP`, `ET`), comparing prediction types across stations.
- `feature_vs_output`: for each feature, two panels (`GPP` and `ET`) with one line per station for a selected prediction type.
- `single_output_all_features`: one station and one output at a time, with subplots for all selected features.

**For `pairwise` (2D ALE):**
- `feature_pair_3D` (default): one 3D surface per station, feature pair, and target (`GPP`/`ET`).
- `feature_pair_interaction`: 2D contour heatmaps per station, feature pair, and target.
- `feature_pair_3D_subplots`: combined 3D subplot grid across stations and prediction types with shared z-axis scaling.
- `feature_pair_3D_subplots_individual`: same subplot layout, but each panel keeps its own z-axis scaling.

### Workflow
1. Choose whether to plot `results` from this notebook run or load a saved bundle, add the path to the saved bundle.
2. Set `plot_mode` and optional filters (`plot_prediction_types`, `plot_stations`, `plot_features`).
3. Run the plotting cell to show and/or save figures.


In [ ]:
# --- Plot config ---

from XAI.scripts.plot_ale import plot_ale_results, to_physical_units


# Quick reference for valid modes
AVAILABLE_PLOT_MODES = {
    "main": [
        "station_vs_prediction",
        "feature_vs_output",
        "single_output_all_features",
    ],
    "pairwise": [
        "feature_pair_3D",
        "feature_pair_interaction",
        "feature_pair_3D_subplots",
    ],
}

# Use in-memory `results` from the ALE run cell when available.
# Set to False to always load the bundle from disk.
use_results_from_notebook = True

plot_variant = variant
plot_length = length
plot_grid_size = grid_size
plot_save_dir_base = save_dir_base
plot_split = "test"             # {"test", "train"}: the data the ALE was computed on

# Plot behavior                 # can leave as it, these are the default settings to show all 
plot_mode = None                # None -> default by variant; otherwise choose from AVAILABLE_PLOT_MODES
plot_prediction_types = None    # e.g., ["naive", "residual"] ; None -> auto
plot_features = []              # e.g., ["Tair", "VPD"] ; [] -> all available
plot_stations = []              # e.g., [5, 12, 14, 21] ; [] -> all available

# Output behavior
show_plots = True
save_plots = True
plots_out_dir = os.path.join(plot_save_dir_base, f"plots_{variant_dir}_{plot_length}_grid{plot_grid_size}")

print("[PLOT CONFIG] variant:", plot_variant)
print("[PLOT CONFIG] length:", plot_length)
print("[PLOT CONFIG] grid_size:", plot_grid_size)
print("[PLOT CONFIG] plot_mode:", plot_mode)
print("[PLOT CONFIG] prediction_types:", plot_prediction_types)
print("[PLOT CONFIG] features:", plot_features if plot_features else "<all>")
print("[PLOT CONFIG] stations:", plot_stations if plot_stations else "<all>")
print("[PLOT CONFIG] save_plots:", save_plots, "->", plots_out_dir if save_plots else "<disabled>")

print("[PLOT CONFIG] available modes for variant:", AVAILABLE_PLOT_MODES.get(plot_variant, "<see script>"))


In [ ]:
# --- Run plotting ---
default_mode_by_variant = {
    "main":     "station_vs_prediction",
    "pairwise": "feature_pair_3D",
    "full":     "feature_pair_3D_subplots_individual",
}

effective_plot_mode = plot_mode or default_mode_by_variant.get(plot_variant, "station_vs_prediction")

if use_results_from_notebook and "results" in globals():
    ale_results_for_plot = results
    print("[PLOT] Using in-memory `results` from this notebook.")
else:
    # To plot a previous run, point `bundle_path` at its <variant>_bundle.joblib here.
    ale_results_for_plot = joblib.load(bundle_path)
    print(f"[PLOT] Loaded bundle: {bundle_path}")

# Model curves in physical units, using the train-fitted constants saved with the run
with open(os.path.join(os.path.dirname(bundle_path), f"{variant}_normalization_stats.json")) as f:
    ale_results_for_plot = to_physical_units(ale_results_for_plot, json.load(f))

available_pred_types = [k for k in ale_results_for_plot.keys() if k != "process"]
if not available_pred_types:
    raise ValueError("No prediction types found in ALE results.")

effective_prediction_types = plot_prediction_types or available_pred_types

save_dir = plots_out_dir if save_plots else None
if save_dir:
    os.makedirs(save_dir, exist_ok=True)

plot_ale_results(
    ale_results=ale_results_for_plot,
    plot_mode=effective_plot_mode,
    features=plot_features if plot_features else None,
    stations=plot_stations if plot_stations else None,
    prediction_type=effective_prediction_types,
    save_dir=save_dir,
    split=plot_split,
    show=show_plots
)

print("[OK] Plotting complete.")
if save_dir:
    print(f"[OK] Plots written to: {save_dir}")
